# Turn-level BLEU/Jaccard vs. Turn-level Grice Coherence (MultiWOZ)

This notebook creates the **turn-level** version of the analysis pipeline.

## Goal
Test the direct version of H1:

> When annotation overlap for a **turn** is low, the Grice-based coherence score for that **turn** is also low.

## What this notebook does
1. Load the original MultiWOZ annotations and your own annotation export.
2. Build a **turn-level overlap table** for `strict`, `normalized`, and `lenient` matching.
3. Load the Grice JSONL/TXT file and aggregate it to **one row per turn**.
4. Merge both tables on `dialog_id + turn_index`.
5. Run first hypothesis tests:
   - Spearman correlations
   - overlap-bin descriptives
   - low-vs-high overlap comparisons

## Important note about `<TURN>`
This notebook **does not** use the synthetic `<TURN>` boundary token.

That token only made sense when entire dialogues were flattened into one long sequence.  
At turn level, each row is already exactly one turn, so adding `<TURN>` would only inject artificial structure.


In [1]:
import json
import re
from collections import defaultdict
from json import JSONDecodeError
from pathlib import Path

import numpy as np
import pandas as pd
from sacrebleu.metrics import BLEU
from scipy.stats import spearmanr, mannwhitneyu


## Paths and configuration

Adjust `GRICE_JSONL_PATH` if your Grice file has a different name or location.


In [12]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
PROC_DIR = DATA_DIR / "processed"
RES_DIR = Path("../results")
RES_DIR.mkdir(parents=True, exist_ok=True)

DIALOGS_PATH = RAW_DIR / "multiwoz_original.json"
ACTS_PATH = RAW_DIR / "dialogue_acts.json"
ANN1_PATH = PROC_DIR / "ann1.txt"
COMMON_IDS_PATH = PROC_DIR / "common_dialog_ids.json"

# Change this if needed.
GRICE_JSONL_PATH = PROC_DIR / "Grice_by_Ben.txt"

# Optional cache paths.
TURN_OVERLAP_OUT = RES_DIR / "turn_level_overlap_no_turn.csv"
GRICE_TURN_OUT = RES_DIR / "grice_turn_level.csv"
MERGED_TURN_OUT = RES_DIR / "merged_turn_analysis.csv"
SPEARMAN_OUT = RES_DIR / "turn_spearman_results.csv"
BIN_SUMMARY_OUT = RES_DIR / "turn_overlap_bin_summary.csv"
GROUP_TESTS_OUT = RES_DIR / "turn_low_vs_high_tests.csv"


## Load source data

In [3]:
with COMMON_IDS_PATH.open("r", encoding="utf-8") as f:
    common_ids = set(json.load(f))

with DIALOGS_PATH.open("r", encoding="utf-8") as f:
    dialogs = json.load(f)

with ACTS_PATH.open("r", encoding="utf-8") as f:
    orig_acts = json.load(f)

print("Common IDs:", len(common_ids))
print("Dialogs loaded:", len(dialogs))
print("Dialogue acts loaded:", len(orig_acts))


Common IDs: 10438
Dialogs loaded: 10438
Dialogue acts loaded: 10438


## Candidate annotation parser

This reuses the parser logic from the dialogue-level notebook.


In [4]:
def parse_annotation_txt(path):
    """Parse the custom TXT annotation export into a MultiWOZ-like dict.

    Each line is expected to look like:
        <utterance>\t<json1>[\t<json2>\t<json3>...]

    Returns:
        dict[str, dict]: Mapping from dialog ID to a dict with `log_by_turn`.
    """
    dialogs = defaultdict(lambda: {"log_by_turn": {}})

    stats = {
        "lines_total": 0,
        "lines_no_json": 0,
        "lines_no_base": 0,
        "lines_no_da": 0,
        "lines_parsed": 0,
        "turn_overwrites": 0,
    }

    with open(path, encoding="utf-8") as f:
        for lineno, raw_line in enumerate(f, start=1):
            stats["lines_total"] += 1

            line = raw_line.rstrip("\n")
            if not line.strip():
                continue

            parts = line.split("\t")
            if len(parts) < 2:
                stats["lines_no_json"] += 1
                continue

            utterance = parts[0]

            json_strs = []
            for part in parts[1:]:
                part = part.strip()
                if not part:
                    continue
                if "{" in part and "}" in part:
                    start = part.index("{")
                    end = part.rfind("}") + 1
                    json_strs.append(part[start:end])

            if not json_strs:
                stats["lines_no_json"] += 1
                continue

            objs = []
            for js in json_strs:
                try:
                    objs.append(json.loads(js))
                except JSONDecodeError as e:
                    print(f"[ERROR] JSON parse failed on line {lineno}: {e}")
                    print("Chunk:", repr(js))
                    raise

            base = None
            for obj in objs:
                if "_dialog_index" in obj and "_turn_index" in obj:
                    base = obj
                    break

            if base is None:
                stats["lines_no_base"] += 1
                continue

            dialog_id = base["_dialog_index"]
            turn_index = int(base["_turn_index"])
            speaker = next((obj.get("speaker") for obj in objs if "speaker" in obj), "unknown")

            dialog_act = {}
            for obj in objs:
                act_name = obj.get("dialog_act")
                if not act_name:
                    continue

                slots = obj.get("slots", {}) or {}
                slot_list = [[k, v] for k, v in slots.items()]
                dialog_act.setdefault(act_name, []).extend(slot_list)

            if not dialog_act:
                stats["lines_no_da"] += 1
                continue

            turn_obj = {
                "text": utterance,
                "dialog_act": dialog_act,
                "speaker": speaker,
            }

            if turn_index in dialogs[dialog_id]["log_by_turn"]:
                stats["turn_overwrites"] += 1

            dialogs[dialog_id]["log_by_turn"][turn_index] = turn_obj
            stats["lines_parsed"] += 1

    print("parse_annotation_txt stats:", stats)
    return dict(dialogs)


cand_data = parse_annotation_txt(ANN1_PATH)
print("Loaded candidate dialogs:", len(cand_data))


parse_annotation_txt stats: {'lines_total': 144441, 'lines_no_json': 1404, 'lines_no_base': 0, 'lines_no_da': 34, 'lines_parsed': 143000, 'turn_overwrites': 0}
Loaded candidate dialogs: 10438


## Ensure original turns carry `dialog_act`

If `multiwoz_original.json` already contains `dialog_act`, this leaves it unchanged.  
If not, it enriches each turn from `dialogue_acts.json`.


In [5]:
def ensure_original_dialog_acts(dialogs, orig_acts):
    """Attach dialogue acts to original MultiWOZ turns when missing.

    Args:
        dialogs: MultiWOZ dialogue dict.
        orig_acts: Original dialogue-act dict keyed by dialog ID and turn index.

    Returns:
        dict: Updated dialogue dict.
    """
    updated = 0
    missing_turn_acts = 0

    for dialog_id, dialog in dialogs.items():
        log = dialog.get("log", []) or []
        acts_for_dialog = orig_acts.get(dialog_id, {}) or {}

        for turn_index, turn in enumerate(log):
            if turn.get("dialog_act"):
                continue

            da = acts_for_dialog.get(str(turn_index), {}) or {}
            if da:
                turn["dialog_act"] = da
                updated += 1
            else:
                turn["dialog_act"] = {}
                missing_turn_acts += 1

    print("Original turns enriched from dialogue_acts.json:", updated)
    print("Original turns still without dialog acts:", missing_turn_acts)
    return dialogs


dialogs = ensure_original_dialog_acts(dialogs, orig_acts)


Original turns enriched from dialogue_acts.json: 0
Original turns still without dialog acts: 3739


## Turn linearization and tokenization helpers

In [6]:
def normalize_slot_value(slot, slot_val, mode="strict"):
    """Normalize a slot value according to the selected matching mode.

    Args:
        slot: Normalized slot name.
        slot_val: Raw slot value.
        mode: One of `strict`, `normalized`, or `lenient`.

    Returns:
        str: Normalized slot value.
    """
    if mode == "strict":
        return "None" if slot_val is None else str(slot_val).strip()

    sval = "" if slot_val is None else str(slot_val).strip().lower()

    if sval in {"none", "null", "nan", "n/a", "?"}:
        sval = ""

    if slot in {"wifi", "internet", "parking"}:
        truthy = {"true", "yes", "y", "1"}
        falsy = {"false", "no", "n", "0"}
        if mode == "lenient":
            truthy |= {"free", "available"}

        if sval in truthy:
            sval = "yes"
        elif sval in falsy:
            sval = "no"

    return sval


def linearize_turn(turn_obj, mode="strict"):
    """Linearize one turn into a canonical symbolic string.

    Args:
        turn_obj: Turn dict with a `dialog_act` field.
        mode: One of `strict`, `normalized`, or `lenient`.

    Returns:
        str: Canonical representation like `act1,act2(slot=a,slot=b)`.
    """
    da = turn_obj.get("dialog_act", {}) or {}
    act_names = list(da.keys())

    if mode not in {"strict", "normalized", "lenient"}:
        raise ValueError(f"Unknown mode: {mode}")

    slot_pairs = []
    for act_name in act_names:
        for slot_name, slot_val in da.get(act_name, []) or []:
            slot = str(slot_name).strip().lower()
            sval = normalize_slot_value(slot, slot_val, mode=mode)
            slot_pairs.append((slot, sval))

    acts_part = ",".join(str(act).strip().lower() for act in act_names)

    slot_pairs.sort(key=lambda x: (x[0], x[1]))
    slots_str = ",".join(f"{slot}={value}" for slot, value in slot_pairs) if slot_pairs else ""

    return f"{acts_part}({slots_str})"


def bleu_tokenize(linear_str):
    """Tokenize a linearized turn string into BLEU/Jaccard tokens.

    Args:
        linear_str: Canonical linearized turn string.

    Returns:
        list[str]: Act and slot=value tokens.
    """
    s = linear_str.strip()

    if "(" in s and ")" in s:
        acts_part, slots_part = s.split("(", 1)
        slots_part = slots_part.rsplit(")", 1)[0]
    else:
        acts_part, slots_part = s, ""

    act_tokens = [token.strip() for token in acts_part.split(",") if token.strip()]
    slot_tokens = [token.strip() for token in slots_part.split(",") if token.strip()]

    tokens = [re.sub(r"\s+", "", token) for token in (act_tokens + slot_tokens)]
    return [token for token in tokens if token]


def jaccard_similarity(tokens_a, tokens_b):
    """Compute Jaccard similarity on token sets.

    Args:
        tokens_a: Token list A.
        tokens_b: Token list B.

    Returns:
        float: Jaccard similarity in [0, 1].
    """
    set_a = set(tokens_a)
    set_b = set(tokens_b)

    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0

    return len(set_a & set_b) / len(set_a | set_b)


def infer_speaker_from_turn_index(turn_index):
    """Infer the default speaker from turn parity.

    MultiWOZ alternates user/system turns, so parity is a useful fallback.

    Args:
        turn_index: Integer turn index.

    Returns:
        str: `customer` for even turns, `expert` for odd turns.
    """
    return "customer" if int(turn_index) % 2 == 0 else "expert"


## Build turn-level tables

One row = one turn.  
No `<TURN>` token is used here because turn boundaries are already explicit in the row structure.


In [7]:
def build_turn_table(data, source_name, mode="strict", keep_only_common_dialogs=True):
    """Build a turn-level symbolic table from original or candidate annotations.

    Args:
        data: Original dialogue dict or candidate dict.
        source_name: Label such as `ref` or `hyp`.
        mode: One of `strict`, `normalized`, or `lenient`.
        keep_only_common_dialogs: Whether to restrict to shared dialog IDs.

    Returns:
        pd.DataFrame: One row per turn.
    """
    rows = []

    for dialog_id, dialog in data.items():
        if keep_only_common_dialogs and dialog_id not in common_ids:
            continue

        if dialog.get("log_by_turn"):
            turn_items = sorted(dialog["log_by_turn"].items())
        else:
            log = dialog.get("log", []) or []
            turn_items = list(enumerate(log))

        for turn_index, turn in turn_items:
            linear = linearize_turn(turn, mode=mode)
            tokens = bleu_tokenize(linear)
            speaker = turn.get("speaker") or infer_speaker_from_turn_index(turn_index)

            rows.append({
                "dialog_id": dialog_id,
                "turn_index": int(turn_index),
                "speaker": str(speaker).lower().strip(),
                f"{source_name}_linear": linear,
                f"{source_name}_tokens": tokens,
                f"{source_name}_token_text": " ".join(tokens),
                f"{source_name}_len": len(tokens),
            })

    df = pd.DataFrame(rows)
    return df.sort_values(["dialog_id", "turn_index"]).reset_index(drop=True)


In [8]:
bleu1_metric = BLEU(max_ngram_order=1, effective_order=True)
bleu4_metric = BLEU(max_ngram_order=4, effective_order=True)

overlap_rows = []

for mode in ["strict", "normalized", "lenient"]:
    print(f"Building turn-level overlap table for mode: {mode}")

    ref_turns = build_turn_table(dialogs, source_name="ref", mode=mode)
    hyp_turns = build_turn_table(cand_data, source_name="hyp", mode=mode)

    merged_turns = ref_turns.merge(
        hyp_turns,
        on=["dialog_id", "turn_index"],
        how="inner",
        suffixes=("_ref", "_hyp"),
    )

    print("Matched turns:", len(merged_turns))

    for row in merged_turns.itertuples(index=False):
        ref_tokens = row.ref_tokens
        hyp_tokens = row.hyp_tokens
        ref_text = row.ref_token_text
        hyp_text = row.hyp_token_text

        if not ref_text and not hyp_text:
            bleu1_turn = 100.0
            bleu4_turn = 100.0
        elif not ref_text or not hyp_text:
            bleu1_turn = 0.0
            bleu4_turn = 0.0
        else:
            bleu1_turn = bleu1_metric.sentence_score(hyp_text, [ref_text]).score
            bleu4_turn = bleu4_metric.sentence_score(hyp_text, [ref_text]).score

        overlap_rows.append({
            "dialog_id": row.dialog_id,
            "turn_index": row.turn_index,
            "mode": mode,
            "speaker_ref": row.speaker_ref,
            "speaker_hyp": row.speaker_hyp,
            "ref_len": row.ref_len,
            "hyp_len": row.hyp_len,
            "jaccard": jaccard_similarity(ref_tokens, hyp_tokens),
            "bleu1_turn": bleu1_turn,
            "bleu4_turn": bleu4_turn,
            "exact_match": int(row.ref_linear == row.hyp_linear),
        })

turn_overlap = pd.DataFrame(overlap_rows).sort_values(
    ["mode", "dialog_id", "turn_index"]
).reset_index(drop=True)

turn_overlap.to_csv(TURN_OVERLAP_OUT, index=False, encoding="utf-8")

print("Turn-level overlap shape:", turn_overlap.shape)
print()
print(turn_overlap.head(10))
print()
print("Mean overlap by mode:")
print(
    turn_overlap.groupby("mode")[["jaccard", "bleu1_turn", "bleu4_turn", "exact_match"]]
    .mean()
)
print()
print("Saved:", TURN_OVERLAP_OUT.resolve())


Building turn-level overlap table for mode: strict
Matched turns: 143000
Building turn-level overlap table for mode: normalized
Matched turns: 143000
Building turn-level overlap table for mode: lenient
Matched turns: 143000
Turn-level overlap shape: (429000, 11)

      dialog_id  turn_index     mode speaker_ref speaker_hyp  ref_len  \
0  MUL0001.json           0  lenient    customer    customer        2   
1  MUL0001.json           1  lenient      expert      expert        6   
2  MUL0001.json           2  lenient    customer    customer        4   
3  MUL0001.json           3  lenient      expert      expert        7   
4  MUL0001.json           4  lenient    customer    customer        4   
5  MUL0001.json           5  lenient      expert      expert        2   
6  MUL0001.json           6  lenient    customer    customer        4   
7  MUL0001.json           7  lenient      expert      expert        9   
8  MUL0001.json           8  lenient    customer    customer        0   
9  MUL

## Load and aggregate Grice JSONL/TXT to turn level

In [13]:
score_map = {
    "Very poor": 1,
    "Poor": 2,
    "Weak": 3,
    "Neutral": 4,
    "Good": 5,
    "Very good": 6,
    "Excellent": 7,
}

if not GRICE_JSONL_PATH.exists():
    raise FileNotFoundError(
        f"Grice file not found: {GRICE_JSONL_PATH}. "
        "Update GRICE_JSONL_PATH in the configuration cell."
    )

grice_rows = []
with GRICE_JSONL_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        if obj.get("score") is None:
            continue

        grice_rows.append({
            "dialog_id": obj.get("_dialog_id"),
            "turn_index": int(obj.get("_turn_index")),
            "unit_index": obj.get("_unit_index"),
            "speaker": str(obj.get("_speaker")).lower().strip() if obj.get("_speaker") is not None else None,
            "score_label": obj.get("score"),
            "score_num": score_map.get(obj.get("score")),
        })

grice_units = pd.DataFrame(grice_rows)

print("Grice unit-level shape:", grice_units.shape)
print(grice_units.head())
print()
print("Score distribution:")
print(grice_units["score_label"].value_counts(dropna=False))


Grice unit-level shape: (234046, 6)
       dialog_id  turn_index  unit_index   speaker score_label  score_num
0  SNG01856.json           0           0  customer        Good          5
1  SNG01856.json           0           1  customer        Good          5
2  SNG01856.json           0           2  customer        Good          5
3  SNG01856.json           1           0    expert        Good          5
4  SNG01856.json           2           0  customer        Good          5

Score distribution:
score_label
Good         140187
Neutral       40640
Excellent     32568
Very good     15008
Poor           2813
Very poor      1551
Weak           1279
Name: count, dtype: int64


In [14]:
grice_turn = (
    grice_units.groupby(["dialog_id", "turn_index", "speaker"], as_index=False)
    .agg(
        grice_turn_mean=("score_num", "mean"),
        grice_turn_median=("score_num", "median"),
        grice_turn_min=("score_num", "min"),
        grice_turn_max=("score_num", "max"),
        grice_n_units=("score_num", "count"),
    )
    .sort_values(["dialog_id", "turn_index"])
    .reset_index(drop=True)
)

grice_turn.to_csv(GRICE_TURN_OUT, index=False, encoding="utf-8")

print("Turn-level Grice shape:", grice_turn.shape)
print()
print(grice_turn.head(10))
print()
print("Summary of mean turn coherence:")
print(grice_turn["grice_turn_mean"].describe())
print()
print("Saved:", GRICE_TURN_OUT.resolve())


Turn-level Grice shape: (142902, 8)

      dialog_id  turn_index   speaker  grice_turn_mean  grice_turn_median  \
0  MUL0001.json           0  customer         5.000000                5.0   
1  MUL0001.json           1    expert         5.500000                5.5   
2  MUL0001.json           2  customer         5.000000                5.0   
3  MUL0001.json           3    expert         4.666667                5.0   
4  MUL0001.json           4  customer         5.000000                5.0   
5  MUL0001.json           5    expert         5.000000                5.0   
6  MUL0001.json           6  customer         4.500000                4.5   
7  MUL0001.json           7    expert         5.666667                5.0   
8  MUL0001.json           8  customer         5.000000                5.0   
9  MUL0001.json           9    expert         5.000000                5.0   

   grice_turn_min  grice_turn_max  grice_n_units  
0               5               5              1  
1            

## Merge overlap and Grice at turn level

In [15]:
merged_turn = turn_overlap.merge(
    grice_turn,
    on=["dialog_id", "turn_index"],
    how="inner",
    suffixes=("", "_grice"),
)

merged_turn["speaker_match"] = (
    merged_turn["speaker_hyp"].fillna("") == merged_turn["speaker"].fillna("")
).astype(int)

merged_turn.to_csv(MERGED_TURN_OUT, index=False, encoding="utf-8")

print("Merged turn-level shape:", merged_turn.shape)
print()
print("Rows per mode:")
print(merged_turn["mode"].value_counts())
print()
print("Unique dialog IDs:", merged_turn["dialog_id"].nunique())
print("Unique turns:", merged_turn[["dialog_id", "turn_index"]].drop_duplicates().shape[0])
print()
print(merged_turn.head(10))
print()
print("Saved:", MERGED_TURN_OUT.resolve())


Merged turn-level shape: (428562, 18)

Rows per mode:
mode
lenient       142854
normalized    142854
strict        142854
Name: count, dtype: int64

Unique dialog IDs: 10438
Unique turns: 142854

      dialog_id  turn_index     mode speaker_ref speaker_hyp  ref_len  \
0  MUL0001.json           0  lenient    customer    customer        2   
1  MUL0001.json           1  lenient      expert      expert        6   
2  MUL0001.json           2  lenient    customer    customer        4   
3  MUL0001.json           3  lenient      expert      expert        7   
4  MUL0001.json           4  lenient    customer    customer        4   
5  MUL0001.json           5  lenient      expert      expert        2   
6  MUL0001.json           6  lenient    customer    customer        4   
7  MUL0001.json           7  lenient      expert      expert        9   
8  MUL0001.json           8  lenient    customer    customer        0   
9  MUL0001.json           9  lenient      expert      expert        2   



## H1 test: Spearman correlations

In [16]:
corr_rows = []

for mode in ["strict", "normalized", "lenient"]:
    sub = merged_turn[merged_turn["mode"] == mode].copy()

    rho_j, p_j = spearmanr(sub["jaccard"], sub["grice_turn_mean"])
    rho_b1, p_b1 = spearmanr(sub["bleu1_turn"], sub["grice_turn_mean"])
    rho_b4, p_b4 = spearmanr(sub["bleu4_turn"], sub["grice_turn_mean"])

    corr_rows.append({
        "mode": mode,
        "n": len(sub),
        "rho_jaccard_grice": rho_j,
        "p_jaccard_grice": p_j,
        "rho_bleu1_grice": rho_b1,
        "p_bleu1_grice": p_b1,
        "rho_bleu4_grice": rho_b4,
        "p_bleu4_grice": p_b4,
    })

turn_corr = pd.DataFrame(corr_rows)
turn_corr.to_csv(SPEARMAN_OUT, index=False, encoding="utf-8")

print(turn_corr)
print()
print("Saved:", SPEARMAN_OUT.resolve())


         mode       n  rho_jaccard_grice  p_jaccard_grice  rho_bleu1_grice  \
0      strict  142854          -0.097680    8.777797e-300         0.167393   
1  normalized  142854          -0.050892     1.482293e-82         0.142603   
2     lenient  142854          -0.050418     4.673171e-81         0.142274   

   p_bleu1_grice  rho_bleu4_grice  p_bleu4_grice  
0            0.0         0.051947   6.111280e-86  
1            0.0        -0.016047   1.314221e-09  
2            0.0        -0.014852   1.978891e-08  

Saved: C:\Users\benwe\Documents\GitHub\multiwoz-grice-agreement\results\turn_spearman_results.csv


## Descriptive check: average Grice score by overlap bins

This makes the pattern easier to interpret than a correlation alone.


In [17]:
bin_rows = []

for mode in ["strict", "normalized", "lenient"]:
    sub = merged_turn[merged_turn["mode"] == mode].copy()

    # Use quintiles for readability.
    sub["jaccard_bin"] = pd.qcut(sub["jaccard"], q=5, duplicates="drop")
    sub["bleu4_bin"] = pd.qcut(sub["bleu4_turn"], q=5, duplicates="drop")

    jacc_summary = (
        sub.groupby("jaccard_bin", observed=False)["grice_turn_mean"]
        .agg(["mean", "median", "count"])
        .reset_index()
    )
    jacc_summary["mode"] = mode
    jacc_summary["metric"] = "jaccard"

    bleu_summary = (
        sub.groupby("bleu4_bin", observed=False)["grice_turn_mean"]
        .agg(["mean", "median", "count"])
        .reset_index()
        .rename(columns={"bleu4_bin": "jaccard_bin"})
    )
    bleu_summary["mode"] = mode
    bleu_summary["metric"] = "bleu4"

    bin_rows.append(jacc_summary.rename(columns={"jaccard_bin": "bin"}))
    bin_rows.append(bleu_summary.rename(columns={"jaccard_bin": "bin"}))

bin_summary = pd.concat(bin_rows, ignore_index=True)
bin_summary.to_csv(BIN_SUMMARY_OUT, index=False, encoding="utf-8")

print(bin_summary.head(20))
print()
print("Saved:", BIN_SUMMARY_OUT.resolve())


                 bin      mean  median  count        mode   metric
0    (-0.001, 0.111]  5.028111     5.0  30215      strict  jaccard
1     (0.111, 0.222]  5.268230     5.0  28029      strict  jaccard
2     (0.222, 0.333]  4.944129     5.0  47498      strict  jaccard
3       (0.333, 0.5]  4.962123     5.0  26767      strict  jaccard
4         (0.5, 1.0]  5.137649     5.0  10345      strict  jaccard
5    (-0.001, 4.457]  5.006486     5.0  28868      strict    bleu4
6    (4.457, 10.208]  5.002968     5.0  28275      strict    bleu4
7   (10.208, 18.394]  4.961634     5.0  30785      strict    bleu4
8   (18.394, 36.362]  5.169907     5.0  26360      strict    bleu4
9    (36.362, 100.0]  5.089447     5.0  28566      strict    bleu4
10   (-0.001, 0.125]  5.019650     5.0  28848  normalized  jaccard
11     (0.125, 0.25]  5.148971     5.0  35361  normalized  jaccard
12     (0.25, 0.333]  4.933053     5.0  34292  normalized  jaccard
13      (0.333, 0.5]  4.996194     5.0  29646  normalized  jac

## Low-vs-high overlap comparison

This compares the bottom quartile and top quartile of overlap for each mode.


In [18]:
group_rows = []

for mode in ["strict", "normalized", "lenient"]:
    sub = merged_turn[merged_turn["mode"] == mode].copy()

    for metric in ["jaccard", "bleu1_turn", "bleu4_turn"]:
        q25 = sub[metric].quantile(0.25)
        q75 = sub[metric].quantile(0.75)

        low = sub.loc[sub[metric] <= q25, "grice_turn_mean"]
        high = sub.loc[sub[metric] >= q75, "grice_turn_mean"]

        stat, p_value = mannwhitneyu(low, high, alternative="two-sided")

        group_rows.append({
            "mode": mode,
            "metric": metric,
            "q25": q25,
            "q75": q75,
            "n_low": len(low),
            "n_high": len(high),
            "low_mean_grice": low.mean(),
            "high_mean_grice": high.mean(),
            "difference_high_minus_low": high.mean() - low.mean(),
            "mannwhitney_u": stat,
            "p_value": p_value,
        })

group_tests = pd.DataFrame(group_rows)
group_tests.to_csv(GROUP_TESTS_OUT, index=False, encoding="utf-8")

print(group_tests)
print()
print("Saved:", GROUP_TESTS_OUT.resolve())


         mode      metric        q25        q75  n_low  n_high  \
0      strict     jaccard   0.142857   0.400000  37433   35942   
1      strict  bleu1_turn   8.208500  58.410059  35766   35829   
2      strict  bleu4_turn   4.978707  29.847459  40381   36419   
3  normalized     jaccard   0.166667   0.428571  38547   36709   
4  normalized  bleu1_turn  13.533528  65.143906  37345   35862   
5  normalized  bleu4_turn   8.208500  31.947155  35780   36650   
6     lenient     jaccard   0.166667   0.428571  37538   37244   
7     lenient  bleu1_turn  13.533528  66.666667  37214   35890   
8     lenient  bleu4_turn   8.400789  32.408078  35722   35731   

   low_mean_grice  high_mean_grice  difference_high_minus_low  mannwhitney_u  \
0        5.090356         4.999424                  -0.090932    735508060.0   
1        4.832909         5.134308                   0.301400    497980275.0   
2        4.890686         5.107632                   0.216946    618353681.0   
3        5.092338  

## Optional next steps

Useful extensions once the basic turn-level relationship is visible:

1. Add a regression predicting `grice_turn_mean` from overlap plus controls:
   - speaker
   - turn index
   - reference length
   - hypothesis length
   - exact-match flag

2. Inspect disagreement slices:
   - low overlap + high coherence
   - high overlap + low coherence

3. Compare whether the pattern differs between `customer` and `expert` turns.
